In [1]:
from datetime import datetime
import requests
import pandas as pd
import numpy as np
import time
# --- HÀM TẢI DỮ LIỆU TỪ BINANCE ---
def str_to_millis(date_str):
    dt = datetime.strptime(date_str, "%d/%m/%Y")
    return int(dt.timestamp() * 1000)

def fetch_binance_kline_df(symbol, interval, start_str, end_str):
    base_url = "https://api.binance.us/"
    api_url = "api/v3/klines"
    general_url = base_url + api_url

    start_time = str_to_millis(start_str)
    end_dt = datetime.strptime(end_str, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
    end_time = int(end_dt.timestamp() * 1000)

    all_klines = []
    while start_time < end_time:
        params = {
            "symbol": symbol.upper(),
            "interval": interval,
            "startTime": start_time,
            "endTime": end_time,
            "limit": 1000
        }
        response = requests.get(general_url, params=params)
        if response.status_code != 200:
            print("Error:", response.status_code, response.text)
            break

        data = response.json()
        if not data:
            break
        all_klines.extend(data)

        last_ts = data[-1][0]
        start_time = last_ts + 1
        time.sleep(0.3)

    df = pd.DataFrame(all_klines, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "num_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ])
    df = df[["timestamp", "open", "close", "high", "low", "volume"]]
    df.columns = ["timestamp", "open", "close", "high", "low", "volume"]
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df[["open", "close", "high", "low", "volume"]] = df[["open", "close", "high", "low", "volume"]].astype(float)
    df["date"] = df["timestamp"].dt.date
    return df


In [2]:
# --- NHẬP THỜI GIAN ---
symbol = "BTCUSDT"
interval = "15m"
start_date = "01/01/2025"
end_date = "30/06/2025"

print("\U0001F4E5 Đang tải dữ liệu từ Binance...")
original_df_all = fetch_binance_kline_df(symbol, interval, start_date, end_date)
print(f"✅ Dữ liệu đã tải: {len(original_df_all)} dòng, từ {original_df_all['date'].min()} đến {original_df_all['date'].max()}")

# --- TÁCH DỮ LIỆU ---
original_daily_data = dict(tuple(original_df_all.groupby('date')))
original_sorted_dates = sorted(original_daily_data.keys())
if len(original_sorted_dates) < 30:
    raise Exception("❌ Cần ít nhất 30 ngày dữ liệu!")

📥 Đang tải dữ liệu từ Binance...
✅ Dữ liệu đã tải: 17376 dòng, từ 2025-01-01 đến 2025-06-30


In [3]:

original_df_all

,timestamp,open,close,high,low,volume,date
0,2025-01-01 00:00:00,93500.00,93500.00,93500.00,93421.15,0.04659,2025-01-01
1,2025-01-01 00:15:00,93500.00,93700.00,93778.40,93499.99,2.47072,2025-01-01
2,2025-01-01 00:30:00,93600.00,93809.31,93809.31,93600.00,0.63185,2025-01-01
3,2025-01-01 00:45:00,93830.46,94361.03,94361.03,93809.43,0.08308,2025-01-01
4,2025-01-01 01:00:00,94171.90,93923.84,94171.90,93752.93,0.32008,2025-01-01
...,...,...,...,...,...,...,...
17371,2025-06-30 22:45:00,107301.79,107056.17,107301.79,107056.17,0.01968,2025-06-30
17372,2025-06-30 23:00:00,107112.97,107241.32,107241.98,107068.15,0.03017,2025-06-30
17373,2025-06-30 23:15:00,107240.44,107064.34,107240.44,107064.34,0.01951,2025-06-30
17374,2025-06-30 23:30:00,107078.20,107144.71,107144.71,107026.24,0.04333,2025-06-30


In [13]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
def calculate_rma(series, period):
    """Calculate Running Moving Average (RMA)"""
    alpha = 1.0 / period
    rma = series.copy()
    for i in range(1, len(series)):
        rma.iloc[i] = alpha * series.iloc[i] + (1 - alpha) * rma.iloc[i-1]
    return rma

def calculate_atr(high, low, close, period=14):
    """Calculate Average True Range (ATR)"""
    tr1 = high - low
    tr2 = abs(high - close.shift(1))
    tr3 = abs(low - close.shift(1))
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(period).mean()

def calculate_adx(high, low, close, period=14):
    """Calculate Average Directional Index (ADX)"""
    # Calculate True Range
    tr = calculate_atr(high, low, close, period)

    # Calculate Directional Movement
    up_move = high.diff()
    down_move = -low.diff()

    plus_dm = pd.Series(0.0, index=high.index)
    minus_dm = pd.Series(0.0, index=high.index)

    # Apply conditions for directional movement
    plus_dm[(up_move > down_move) & (up_move > 0)] = up_move[(up_move > down_move) & (up_move > 0)]
    minus_dm[(down_move > up_move) & (down_move > 0)] = down_move[(down_move > up_move) & (down_move > 0)]

    # Smooth DM and TR
    plus_dm_smooth = calculate_rma(plus_dm, period)
    minus_dm_smooth = calculate_rma(minus_dm, period)

    # Calculate DI, handle division by zero
    tr_safe = tr.where(tr > 0, 0.01 * tr.mean() if tr.mean() > 0 else 0.01)  # Use small fraction of mean TR
    plus_di = 100 * plus_dm_smooth / tr_safe
    minus_di = 100 * minus_dm_smooth / tr_safe

    # Calculate DX, handle division by zero
    di_sum = plus_di + minus_di
    di_diff = abs(plus_di - minus_di)
    dx = 100 * di_diff / di_sum.where(di_sum > 0, 0.01 * di_sum.mean() if di_sum.mean() > 0 else 0.01)

    # Calculate ADX
    adx = calculate_rma(dx, period)

    # Fill NaNs only for initial periods where calculations are incomplete
    adx = adx.fillna(method='bfill')

    return adx

def calculate_sma(series, period):
    return series.rolling(window=period, min_periods=period).mean()

def calculate_rsi(close, period):
    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period, min_periods=period).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=period, min_periods=period).mean()
    rs = gain / (loss + 1e-10)
    return 100 - (100 / (1 + rs))

def calculate_avg_turnover(close, volume, period):
    return (close * volume).rolling(window=period, min_periods=period).mean()

def run_strategy(data, adx_limit=20.5, intraday_drop_pct=4.0, profit_target_pct=1.2,
                sma_period=100, atr_period=10, rsi_period=3, volume_period=50,
                timeout_days=6, max_open_positions=10, max_orders=30, initial_cash=100000):
    data = data.copy()
    cash = initial_cash
    positions = []
    portfolio_values = []
    market_returns = []
    strategy_returns = []

    # Calculate indicators
    data['avg_volume'] = calculate_sma(data['volume'], volume_period)
    data['avg_turnover'] = calculate_avg_turnover(data['close'], data['volume'], volume_period)
    data['atr'] = calculate_atr(data['high'], data['low'], data['close'], atr_period)
    data['atrp'] = (data['atr'] / data['close']) * 100
    data['sma100'] = calculate_sma(data['close'], sma_period)
    data['adx7'] = calculate_adx(data['high'], data['low'], data['close'], period=7)
    data['rsi3'] = calculate_rsi(data['close'], rsi_period)
    # print(data)
    open_positions = []
    candidates = []

    for idx in range(sma_period, len(data)):
        date = data.index[idx]
        close = data['close'].iloc[idx]
        prev_close = data['close'].iloc[idx-1] if idx > 0 else np.nan

        # Calculate market return
        market_return = (close / prev_close - 1) if idx > 0 and not np.isnan(prev_close) else 0
        market_returns.append(market_return)

        # Update positions and calculate strategy return
        portfolio_value = cash
        position_return = 0
        position = 1 if open_positions else 0  # 1 if holding position, 0 otherwise
        for pos in open_positions[:]:
            current_price = close
            pos['days_held'] += 1
            # Exit conditions
            if pos['days_held'] >= timeout_days:
                cash += pos['shares'] * current_price
                position_return += (current_price / pos['entry_price'] - 1) * pos['shares']
                open_positions.remove(pos)
                continue
            stop_price = pos['entry_price'] - 3.0 * data['atr'].iloc[pos['entry_bar']]
            profit_price = pos['entry_price'] + profit_target_pct * data['atr'].iloc[idx]
            if current_price <= stop_price:
                cash += pos['shares'] * stop_price
                position_return += (stop_price / pos['entry_price'] - 1) * pos['shares']
                open_positions.remove(pos)
                continue
            if current_price >= profit_price:
                cash += pos['shares'] * profit_price
                position_return += (profit_price / pos['entry_price'] - 1) * pos['shares']
                open_positions.remove(pos)
                continue
            portfolio_value += pos['shares'] * current_price

        # Strategy return = market return * position
        strategy_return = market_return * position
        strategy_returns.append(strategy_return)
        portfolio_values.append(portfolio_value)

        # Entry logic
        if len(open_positions) >= max_open_positions:
            continue

        if (np.any(data[['sma100', 'atrp', 'atr', 'avg_volume', 'rsi3', 'adx7', 'avg_turnover']].iloc[idx].isna())):
            continue

        if (data['avg_volume'].iloc[idx] < 500000 or
            data['avg_turnover'].iloc[idx] < 2.5e6 or
            data['atrp'].iloc[idx] < 4.0 or
            data['adx7'].iloc[idx] <= adx_limit or
            data['Close'].iloc[idx] < data['sma100'].iloc[idx] + data['atr'].iloc[idx] or
            data['rsi3'].iloc[idx] > 50):
            continue

        limit_price = prev_close * (1.0 - intraday_drop_pct / 100.0)
        if close <= limit_price:
            shares = (cash * 0.1) // limit_price  # Risk 10% of cash per trade
            if shares > 0 and len(candidates) < max_orders:
                cash -= shares * limit_price
                open_positions.append({
                    'entry_price': limit_price,
                    'shares': shares,
                    'entry_bar': idx,
                    'days_held': 0
                })
                candidates.append({'rank': data['adx7'].iloc[idx], 'limit_price': limit_price})

    # Calculate cumulative returns
    market_cum_returns = np.cumprod(1 + np.array(market_returns))
    strategy_cum_returns = np.cumprod(1 + np.array(strategy_returns))

    # Plot results
    plt.figure(figsize=(12, 6))
    plt.plot(data.index[sma_period:], market_cum_returns, label='Market Cumulative Return')
    plt.plot(data.index[sma_period:], strategy_cum_returns, label='Strategy Cumulative Return')
    plt.title('Cumulative Returns: Market vs Strategy')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend()
    plt.grid()
    plt.show()

    return {
        'portfolio_values': portfolio_values,
        'market_returns': market_returns,
        'strategy_returns': strategy_returns,
        'market_cum_returns': market_cum_returns,
        'strategy_cum_returns': strategy_cum_returns
    }

In [14]:
run_strategy(original_df_all)

                timestamp       open      close       high        low  \
0     2025-01-01 00:00:00   93500.00   93500.00   93500.00   93421.15   
1     2025-01-01 00:15:00   93500.00   93700.00   93778.40   93499.99   
2     2025-01-01 00:30:00   93600.00   93809.31   93809.31   93600.00   
3     2025-01-01 00:45:00   93830.46   94361.03   94361.03   93809.43   
4     2025-01-01 01:00:00   94171.90   93923.84   94171.90   93752.93   
...                   ...        ...        ...        ...        ...   
17371 2025-06-30 22:45:00  107301.79  107056.17  107301.79  107056.17   
17372 2025-06-30 23:00:00  107112.97  107241.32  107241.98  107068.15   
17373 2025-06-30 23:15:00  107240.44  107064.34  107240.44  107064.34   
17374 2025-06-30 23:30:00  107078.20  107144.71  107144.71  107026.24   
17375 2025-06-30 23:45:00  107080.00  107214.96  107242.13  107011.11   

        volume        date  avg_volume  avg_turnover      atr      atrp  \
0      0.04659  2025-01-01         NaN          

/tmp/ipython-input-1879551678.py:54: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  adx = adx.fillna(method='bfill')
